# caban — pipeline driver

Notebook entry point for the caban analysis pipeline. Each cell runs one stage; you can stop at any point and inspect state in the live kernel.

**Architecture**
- `caban.config.PipelineConfig` — all user-tunable switches.
- `caban.loader.load_all_mice(cfg)` — returns a `SimpleNamespace` (`ds`) with every metadata dict, session dict, mapping accumulator, and engram-pass result.
- `caban.sections` — one `run_<name>(ds, cfg, ...)` function per top-level analysis. Each cell below calls one.
- `caban.pipeline.*` — thin wrappers around the heavier analysis modules (`caban.population`, `caban.isomap`, `caban.epoch_analysis`, ...).

**Live debugging.** To iterate on a section's body, open `caban/sections.py`, find the corresponding `run_<name>` function, copy its body (including the ds/cfg unpacking block) into a scratch cell, and run it inline. `ds`, `cfg`, and any upstream cross-section state are already in scope.


## 1. Setup

    "`%autoreload 2` rebuilds modules on edit. Heavy code lives in `caban/*.py` and is reloaded automatically; the dataset itself is **not** rebuilt — you keep your loaded `ds`."

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, importlib
import numpy as np

import matplotlib
matplotlib.use("Agg") # disable in-line plotting in Jupyter notebook.
import matplotlib.pyplot as plt

import caban.config
import caban.loader
import caban.pipeline
from caban.config import PipelineConfig
from caban.loader import load_all_mice
from caban.pipeline import (
    resolve_continuity_params,
    engram_idx_by_mouse,
    run_engram_sanity_plots,
    run_population_pca,
    run_population_pca_all_modes,
    run_isomap,
    run_epoch_pv,
    run_cross_session_epoch_pv,
)

## 2. Build config

Edit fields here to override defaults. All ~90 switches from `caban/main.py` are exposed.


In [ ]:
cfg = PipelineConfig(
    # DEBUG=True,                # restrict to one mouse for quick smoke tests
    # plot_pf_raw_maps=True,     # heavy per-cell PF plots
    # optimize_parameters=True,  # run raw-S decoder Optuna study
    # enable_population_curve=True,
)
print(cfg)

## 3. Load all mice (with optional pickle cache)

`load_all_mice(cfg)` builds CrossReg + Session objects, runs the per-mapping accumulators, the unified engram-identity pass, and the merged-PF backfill. This is the slow step.

`load_all_mice` handles caching internally:
- If `<NPY_SAVE_PATH>/ds_cache.pkl` exists → loads `ds` from pickle and skips the fresh build.
- Otherwise → builds `ds` fresh and writes the pickle for next time.
- Always prints a deep-size memory breakdown after load.

Pass `use_cache=False` to force a rebuild without touching the pickle; pass `cache_path=...` to override the default location; pass `report_memory=False` to suppress the size breakdown.


In [ ]:
ds = load_all_mice(cfg)

In [ ]:
#globals().update(vars(ds))

## Analysis sections

The cells below follow the exact top-level pathway of `caban/main.py`, one section per cell, in the same order. Each cell calls a `run_<name>(ds, cfg, ...)` function from `caban.sections`.

Every block is gated by its own `cfg.plot_*` / `cfg.enable_*` / `cfg.optimize_*` switch. Toggle on `cfg` and re-run the cell — no globals re-sync needed.


In [ ]:
# Section functions live in caban.sections.
# Each takes (ds, cfg) plus any cross-section state via kwargs
# and is gated by its cfg.plot_* / enable_* switch.
from caban.sections import (
    run_sp_rates,
    run_binned_sp_rates,
    run_ROIs,
    run_proportional_activities,
    run_LT_firing_rate_changes,
    run_PSTH,
    run_pf_and_loc,
    run_occupancy_analysis,
    run_LT_pfs,
    run_LT_decoding,
    run_zone_crossreg,
    run_continuity_and_paramsets,
    run_optimize_raw_decoder,
    run_optimize_pf_decoder,
    run_paradigm_A,
    run_paradigm_B,
    run_paradigm_C,
    run_paradigm_D1,
    run_paradigm_D2,
    run_paradigm_E1,
    run_paradigm_E2,
    run_paradigm_F,
    run_mixedlm_cross_vs_within,
    run_mixedlm_vs_tfc_cond,
    run_pv_correlation_2d,
    run_epoch_pv_within,
    run_epoch_pv_cross,
    run_population_pca,
    run_isomap,
    run_umap,
    run_population_vectors,
    run_population_vector_distances,
    run_binned_activities,
)

# Shared state threaded between sections (None until produced).
raw_params = pf_params = None
lt_cont_pvt = None; use_PCT_error = None
mt_A_results_2D = mt_B_results_2D = mt_C_results_2D = None


### 9. Spike-rate panels

`caban/main.py` L1474–1539


In [ ]:
run_sp_rates(ds, cfg)


### 10. Binned spike-rate panels

`caban/main.py` L1540–1590


In [ ]:
run_binned_sp_rates(ds, cfg)


### 11. ROI maps

`caban/main.py` L1591–1598


In [ ]:
run_ROIs(ds, cfg)


### 12. Proportional activities

`caban/main.py` L1599–1609


In [ ]:
run_proportional_activities(ds, cfg)


### 13. LT firing-rate changes

`caban/main.py` L1610–1646


In [ ]:
run_LT_firing_rate_changes(ds, cfg)


### 14. PSTH

`caban/main.py` L1647–1681


In [ ]:
run_PSTH(ds, cfg)


### 15. Place fields and locations

`caban/main.py` L1682–1779


In [ ]:
run_pf_and_loc(ds, cfg)
# 5m 54.0s

### 16. Occupancy / trajectory / immobility

`caban/main.py` L1780–1818


In [ ]:
run_occupancy_analysis(ds, cfg)
# 1m 47.9s

### 17. LT place fields

`caban/main.py` L1819–2095


In [ ]:
# Temporary insurance to prevent fm loading from pickles crashing since they don't know about SSTCa2->caban rename.
import sys
import caban.spatial
sys.modules['SSTCa2_spatial'] = caban.spatial

In [ ]:
run_LT_pfs(ds, cfg)
# 59m 4.6s

In [ ]:
from caban.loader import report_ds_memory
report_ds_memory(ds)

### 18. LT decoding

`caban/main.py` L2096–2888


In [ ]:
_result = run_LT_decoding(ds, cfg)
lt_cont_pvt   = _result.get("lt_cont_pvt") if _result else None
use_PCT_error = _result.get("use_PCT_error") if _result else None


### 19. Zone cross-registration suite

`caban/main.py` L2889–3037


In [ ]:
run_zone_crossreg(ds, cfg, lt_cont_pvt=lt_cont_pvt, use_PCT_error=use_PCT_error)

# 6m 36.7s

### 20. Decoder constants and continuity sigmas

`caban/main.py` L3038–3215


In [ ]:
_result = run_continuity_and_paramsets(ds, cfg)
raw_params = _result["raw_params"] if _result else None
pf_params  = _result["pf_params"]  if _result else None

# 28.3s 

### 21. Optimize raw-decoder parameters

`caban/main.py` L3216–3285


In [ ]:
run_optimize_raw_decoder(ds, cfg)
# normally should not run as optimize_parameters=False

### 22. Optimize PF-decoder parameters

`caban/main.py` L3286–3537


In [ ]:
run_optimize_pf_decoder(ds, cfg)
# normally should not run as optimize_parameters=False

### 23. 2D Bayesian decoder paradigm A

`caban/main.py` L3783–4016


In [ ]:
_result = run_paradigm_A(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_A_results_2D = _result["mt_A_results_2D"] if _result else None

# 18m 32.7s

### 24. 2D Bayesian decoder paradigm B

`caban/main.py` L4017–4171


In [ ]:
_result = run_paradigm_B(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_B_results_2D = _result["mt_B_results_2D"] if _result else None

# 15m 14.0s

### 25. 2D Bayesian decoder paradigm C

`caban/main.py` L4172–4327


In [ ]:
_result = run_paradigm_C(ds, cfg, raw_params=raw_params, pf_params=pf_params)
mt_C_results_2D = _result["mt_C_results_2D"] if _result else None

# 15m 25.3s

### 26. 2D Bayesian decoder paradigm D1

`caban/main.py` L4328–4477


In [ ]:
run_paradigm_D1(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 11m 4.8s

### 27. 2D Bayesian decoder paradigm D2

`caban/main.py` L4478–4627


In [ ]:
run_paradigm_D2(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 10m 47.7s

### 28. 2D Bayesian decoder paradigm E1

`caban/main.py` L4628–4775


In [ ]:
run_paradigm_E1(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 10m 58.2s

### 29. 2D Bayesian decoder paradigm E2

`caban/main.py` L4776–4924


In [ ]:
run_paradigm_E2(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 10m 38.1s

### 30. 2D Bayesian decoder paradigm F

`caban/main.py` L4925–5127


In [ ]:
run_paradigm_F(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 10m 21.8s

### 31. TFC cross-session vs within-session MixedLM

`caban/main.py` L5128–5155


In [ ]:
run_mixedlm_cross_vs_within(ds, cfg, mt_A_results_2D=mt_A_results_2D, mt_B_results_2D=mt_B_results_2D, mt_C_results_2D=mt_C_results_2D)

# 2m 45.7s

### 32. TFC vs TFC_cond baseline MixedLM

`caban/main.py` L5156–5181


In [ ]:
run_mixedlm_vs_tfc_cond(ds, cfg, mt_A_results_2D=mt_A_results_2D, mt_B_results_2D=mt_B_results_2D, mt_C_results_2D=mt_C_results_2D)

# 2m 39.4s

### 33. 2D Population Vector (PV) correlation

`caban/main.py` L5182–5562


In [ ]:
run_pv_correlation_2d(ds, cfg, raw_params=raw_params, pf_params=pf_params)

# 70m 47.0s

### 34. Within-session epoch-PV similarity

`caban/main.py` L5563–5631


In [ ]:
run_epoch_pv_within(ds, cfg)

# 17m 14.1s

### 35. Cross-session epoch-PV similarity

`caban/main.py` L5632–5708


In [ ]:
run_epoch_pv_cross(ds, cfg)

# >44m

### 36. Population PCA trajectory analysis (incl. engram sanity)

`caban/main.py` L5709–5861


In [ ]:
run_population_pca(ds, cfg)

# 113m 51.6s

In [ ]:
mapping_tfc_b_b1wk = (
    ds.mapping_TFC_cond_Test_B_Test_B_1wk
    if hasattr(ds, 'mapping_TFC_cond_Test_B_Test_B_1wk')
    else 'TFC_cond+Test_B+Test_B_1wk'
)

crossreg_by_mouse = ds.TFC_B_B_1wk_crossreg

mice = sorted(set(ds.TFC_cond) & set(ds.Test_B) & set(ds.Test_B_1wk))
if not mice:
    raise RuntimeError('No mice found in the intersection of TFC_cond, Test_B, and Test_B_1wk.')

rows = []
for mouse in mice:
    if mouse not in crossreg_by_mouse:
        raise RuntimeError(f'Missing crossreg object for mouse {mouse}.')

    n_tfc = ds.TFC_cond[mouse].S.shape[0]
    n_b = ds.Test_B[mouse].S.shape[0]
    n_b1wk = ds.Test_B_1wk[mouse].S.shape[0]

    if n_tfc <= 0:
        raise RuntimeError(f'{mouse}: TFC_cond has zero cells; cannot compute crossreg percentage.')

    df_map = crossreg_by_mouse[mouse].get_mappings_cells(mapping_type=mapping_tfc_b_b1wk)
    n_crossreg = len(df_map)
    pct_crossreg_of_tfc = 100.0 * n_crossreg / n_tfc

    rows.append((mouse, n_tfc, n_b, n_b1wk, n_crossreg, pct_crossreg_of_tfc))

sum_tfc = sum(r[1] for r in rows)
sum_b = sum(r[2] for r in rows)
sum_b1wk = sum(r[3] for r in rows)
sum_crossreg = sum(r[4] for r in rows)
mean_pct = sum(r[5] for r in rows) / len(rows)

if sum_tfc <= 0:
    raise RuntimeError('Total TFC_cond cell count is zero; cannot compute TOTAL percentage.')

pct_total = 100.0 * sum_crossreg / sum_tfc

headers = ('mouse', 'TFC_cond', 'Test_B', 'Test_B_1wk', 'crossreg', '%crossreg_of_TFC_cond')

display_rows = [
    (r[0], str(r[1]), str(r[2]), str(r[3]), str(r[4]), f"{r[5]:.2f}%")
    for r in rows
]

total_row = ('TOTAL', str(sum_tfc), str(sum_b), str(sum_b1wk), str(sum_crossreg), f"{pct_total:.2f}%")
mean_row = (
    'MEAN',
    f"{sum_tfc / len(rows):.2f}",
    f"{sum_b / len(rows):.2f}",
    f"{sum_b1wk / len(rows):.2f}",
    f"{sum_crossreg / len(rows):.2f}",
    f"{mean_pct:.2f}%",
)

widths = [
    max(len(headers[i]), max(len(row[i]) for row in display_rows), len(total_row[i]), len(mean_row[i]))
    for i in range(len(headers))
]

def format_row(values):
    return (
        f"{values[0]:<{widths[0]}}  "
        f"{values[1]:>{widths[1]}}  "
        f"{values[2]:>{widths[2]}}  "
        f"{values[3]:>{widths[3]}}  "
        f"{values[4]:>{widths[4]}}  "
        f"{values[5]:>{widths[5]}}"
    )

line_w = sum(widths) + 2 * (len(widths) - 1)

print(f"Mapping used for crossreg count: {mapping_tfc_b_b1wk}")
print(format_row(headers))
print('-' * line_w)

for row in display_rows:
    print(format_row(row))

print('-' * line_w)
print(format_row(total_row))
print(format_row(mean_row))

### 37. Isomap manifold pipeline

`caban/main.py` L5862–5882


In [ ]:
run_isomap(ds, cfg)

# 213m 26.9s

### 38. UMAP manifold pipeline

`caban/sections.py` run_umap(ds, cfg)

run_umap(ds, cfg)


### 39. Population vector distances

`caban/main.py` L5938–6367


In [ ]:
run_population_vector_distances(ds, cfg)


### 40. Binned activities

`caban/main.py` L6368–6371


In [ ]:
run_binned_activities(ds, cfg)


## Reloading code after edits

`%autoreload 2` (cell 1) handles most edits. If you re-bind names with `from X import Y`, run this cell to refresh them too.
Reload **bottom-up** (dependencies first).

In [ ]:
import importlib
import caban.utilities, caban.sessions, caban.analysis
import caban.engram, caban.engram_sanity
import caban.population, caban.isomap, caban.epoch_analysis, caban.spatial
import caban.decoder
import caban.config, caban.loader, caban.pipeline
for _m in (caban.utilities, caban.sessions, caban.analysis,
           caban.engram, caban.engram_sanity,
           caban.population, caban.isomap, caban.epoch_analysis, caban.spatial,
           caban.decoder,
           caban.config, caban.loader, caban.pipeline):
    importlib.reload(_m)

# Re-import names that were rebound via `from X import Y`:
from caban.config import PipelineConfig
from caban.loader import load_all_mice
from caban.pipeline import (
    resolve_continuity_params, engram_idx_by_mouse, run_engram_sanity_plots,
    run_population_pca, run_population_pca_all_modes,
    run_isomap, run_epoch_pv, run_cross_session_epoch_pv,
)
print('reload complete')